# Sales Data: Exploratory Data Analysis

**Author:** Dr. Saloni Molasi

## Objective
Understand **who buys, what they buy, and where**, so marketing and stocking decisions can be based on data.

## Questions explored
1. Which gender and age groups drive transactions and sales?
2. Which states generate the most orders and revenue?
3. Which product categories, professions, and products matter most?

## Tools
Python, pandas, matplotlib, seaborn

## Dataset
`Sales_Data.csv`: customer transactions with demographics (gender, age, profession), location (state), product details, number of orders, and sales amount.

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


def millions_axis(ax, axis="y"):
    """Show large amounts as 10M, 20M ... instead of 1e7."""
    target = ax.yaxis if axis == "y" else ax.xaxis
    target.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1e6:g}M"))


def add_labels(ax, fmt="{:,.0f}", millions=False, fontsize=9):
    """Write the value on top of every bar (optionally in millions)."""
    for container in ax.containers:
        if millions:
            labels = [f"{v / 1e6:.1f}M" for v in container.datavalues]
            ax.bar_label(container, labels=labels, padding=2, fontsize=fontsize)
        else:
            ax.bar_label(container, fmt=fmt, padding=2, fontsize=fontsize)
    if millions and ax.containers:
        millions_axis(ax, "y" if ax.containers[0].orientation == "vertical" else "x")
    ax.margins(x=0.1, y=0.1)   # leave room so labels are not clipped


def share_insight(table, group_col, value_col, what):
    """Print the top group and its share of the total, so the text always matches the data."""
    top = table.sort_values(value_col, ascending=False).iloc[0]
    share = top[value_col] / table[value_col].sum()
    print(f"Insight: {top[group_col]} leads in {what} with {top[value_col]:,.0f} ({share:.1%} of the total).")

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv("Sales_Data.csv", encoding="unicode_escape")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head(10)

In [ ]:
df.info()

## 3. Data cleaning

Steps taken:
1. Check missing values in every column
2. Remove fully empty rows and columns, and rows with no sales amount
3. Standardise gender labels (`M` to `Male`)
4. Clean hidden characters (non-breaking spaces) in text columns, such as state and category names

In [ ]:
missing = df.isnull().sum()
missing[missing > 0]

In [ ]:
rows_before = len(df)
df = df.dropna(how="all").dropna(axis=1, how="all")   # drop empty rows and empty columns
df = df.dropna(subset=["Amount"])                      # a sale without an amount cannot be analysed
print(f"Rows removed: {rows_before - len(df)} | Rows remaining: {len(df):,}")

In [ ]:
# Replace hidden non-breaking spaces (e.g. "Andhra Pradesh", "Sports") and trim spaces in all text columns
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == "object":
        df[col] = df[col].str.replace("\xa0", " ", regex=False).str.strip()

# Standardise gender labels
df["Gender"] = df["Gender"].replace({"M": "Male", "F": "Female"})

print(df["Gender"].value_counts(), "\n")
print(sorted(df["State"].unique()))
print(sorted(df["Product_Category"].unique()))

## 4. Summary statistics

In [ ]:
df[["Age", "Orders", "Amount"]].describe().round(1)

## 5. Gender analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: number of transactions
sns.countplot(data=df, x="Gender", hue="Gender", legend=False, ax=axes[0])
axes[0].set_title("Transactions by Gender")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Number of transactions")
add_labels(axes[0])

# Right: total sales
gender_sales = (df.groupby("Gender", as_index=False)["Amount"].sum()
                  .sort_values("Amount", ascending=False))
sns.barplot(data=gender_sales, x="Gender", y="Amount", hue="Gender", legend=False, ax=axes[1])
axes[1].set_title("Total Sales Amount by Gender")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Total sales amount")
add_labels(axes[1], millions=True)

plt.tight_layout()
plt.show()

gender_counts = df["Gender"].value_counts().rename_axis("Gender").reset_index(name="Transactions")
share_insight(gender_counts, "Gender", "Transactions", "transactions")
share_insight(gender_sales, "Gender", "Amount", "sales")

## 6. Age group analysis

In [ ]:
age_order = sorted(df["Age Group"].unique())   # keeps age groups in natural order

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: transactions by age group, split by gender
sns.countplot(data=df, x="Age Group", hue="Gender", order=age_order, ax=axes[0])
axes[0].set_title("Transactions by Age Group and Gender")
axes[0].set_xlabel("Age group")
axes[0].set_ylabel("Number of transactions")
add_labels(axes[0], fontsize=8)

# Right: total sales by age group
age_sales = (df.groupby("Age Group", as_index=False)["Amount"].sum()
               .sort_values("Amount", ascending=False))
sns.barplot(data=age_sales, x="Age Group", y="Amount", order=age_order, color="steelblue", ax=axes[1])
axes[1].set_title("Total Sales Amount by Age Group")
axes[1].set_xlabel("Age group")
axes[1].set_ylabel("Total sales amount")
add_labels(axes[1], millions=True, fontsize=8)

plt.tight_layout()
plt.show()

age_counts = df["Age Group"].value_counts().rename_axis("Age Group").reset_index(name="Transactions")
share_insight(age_counts, "Age Group", "Transactions", "transactions")
share_insight(age_sales, "Age Group", "Amount", "sales")

## 7. State analysis (top 5)

In [ ]:
state_orders = (df.groupby("State", as_index=False)["Orders"].sum()
                  .sort_values("Orders", ascending=False))
state_sales = (df.groupby("State", as_index=False)["Amount"].sum()
                 .sort_values("Amount", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=state_orders.head(5), x="State", y="Orders", color="teal", ax=axes[0])
axes[0].set_title("Top 5 States by Number of Orders")
axes[0].set_xlabel("State")
axes[0].set_ylabel("Total orders")
add_labels(axes[0])

sns.barplot(data=state_sales.head(5), x="State", y="Amount", color="darkorange", ax=axes[1])
axes[1].set_title("Top 5 States by Sales Amount")
axes[1].set_xlabel("State")
axes[1].set_ylabel("Total sales amount")
add_labels(axes[1], millions=True)

plt.tight_layout()
plt.show()

share_insight(state_orders, "State", "Orders", "orders")
share_insight(state_sales, "State", "Amount", "sales")
print("Top 5 states by sales:", ", ".join(state_sales.head(5)["State"]))

## 8. Product category analysis

In [ ]:
cat_sales = (df.groupby("Product_Category", as_index=False)["Amount"].sum()
               .sort_values("Amount", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: all categories by sales (up to 10)
sns.barplot(data=cat_sales.head(10), y="Product_Category", x="Amount", color="mediumpurple", ax=axes[0])
axes[0].set_title("Product Categories by Sales")
axes[0].set_xlabel("Total sales amount")
axes[0].set_ylabel("Product category")
add_labels(axes[0], millions=True)

# Right: top 5 categories, split by gender
top_cats = cat_sales.head(5)["Product_Category"].tolist()
cat_gender = (df[df["Product_Category"].isin(top_cats)]
              .groupby(["Product_Category", "Gender"], as_index=False)["Amount"].sum())
sns.barplot(data=cat_gender, x="Product_Category", y="Amount", hue="Gender", order=top_cats, ax=axes[1])
axes[1].set_title("Top 5 Categories: Sales by Gender")
axes[1].set_xlabel("Product category")
axes[1].set_ylabel("Total sales amount")
axes[1].tick_params(axis="x", rotation=20)
millions_axis(axes[1], "y")
axes[1].margins(y=0.15)
axes[1].legend(title="Gender", loc="upper right", ncol=2)

plt.tight_layout()
plt.show()

share_insight(cat_sales, "Product_Category", "Amount", "sales")

## 9. Profession analysis

In [ ]:
prof_sales = (df.groupby("Profession", as_index=False)["Amount"].sum()
                .sort_values("Amount", ascending=False))

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=prof_sales.head(5), x="Profession", y="Amount", color="seagreen")
ax.set_title("Top 5 Professions by Sales Amount")
ax.set_xlabel("Profession")
ax.set_ylabel("Total sales amount")
add_labels(ax, millions=True)
plt.tight_layout()
plt.show()

share_insight(prof_sales, "Profession", "Amount", "sales")
print("Top 3 professions by sales:", ", ".join(prof_sales.head(3)["Profession"]))

## 10. Top 10 products by orders

In [ ]:
product_orders = (df.groupby("Product_ID", as_index=False)["Orders"].sum()
                    .sort_values("Orders", ascending=False))

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=product_orders.head(10), y="Product_ID", x="Orders", color="indianred")
ax.set_title("Top 10 Products by Number of Orders")
ax.set_xlabel("Total orders")
ax.set_ylabel("Product ID")
add_labels(ax)
plt.tight_layout()
plt.show()

## 11. Key findings

In [ ]:
total_sales = df["Amount"].sum()
print(f"Total sales amount analysed: {total_sales:,.0f} across {len(df):,} transactions\n")

summary = [
    ("Gender", gender_sales, "Gender"),
    ("Age group", age_sales, "Age Group"),
    ("State", state_sales, "State"),
    ("Product category", cat_sales, "Product_Category"),
    ("Profession", prof_sales, "Profession"),
]
for label, table, col in summary:
    top = table.sort_values("Amount", ascending=False).iloc[0]
    print(f"Top {label} by sales: {top[col]} ({top['Amount'] / total_sales:.1%} of total)")

### What this means for the business
- **Target the leading customer segment** (the top gender and age group above) with campaigns built around what they buy most.
- **Stock and promote the top product categories** and best-selling products, especially in the highest-selling states.
- **Run region-specific campaigns** in the top states, and test offers in lower-performing states to find growth.
- **Use profession data** to partner with or advertise to the groups that spend the most.

### Limitations
- This is descriptive analysis of one dataset. It shows what happened, not why.
- There is no date column, so seasonality and trends over time could not be examined.
- Differences between groups show association, not cause.

### Next steps
- Add a time dimension to study seasonal patterns.
- Build a dashboard (Power BI) from the cleaned data.
- Test whether the differences between groups are statistically significant.